In [1]:
import pandas as pd
import numpy as np

In [14]:
import os
import sys
import pandas as pd
import pandas_market_calendars as mcal
from datetime import datetime, date, timedelta

# 获取 NYSE 日历# NYSE 是 New York Stock Exchange（纽约证券交易所） 的缩写
calendar = mcal.get_calendar('NYSE')

def is_trading_date(input_date):
    # 获取今天日期（注意时区）
    today = datetime.today().date()

    # 判断是否为交易日
    schedule = calendar.schedule(start_date=today, end_date=today)
    is_trading_day = not schedule.empty

    return is_trading_day


def is_weekly_last_trading_date(input_date):
    if isinstance(input_date, str):
        input_date = pd.to_datetime(input_date).date()
    elif isinstance(input_date, pd.Timestamp):
        input_date = input_date.date()

    # 找到 input_date 所在的那一周（周一至周日）
    year, week_num, weekday = input_date.isocalendar()

    # 构造这一周的日期范围
    start_of_week = date.fromisocalendar(year, week_num, 1)  # 周一
    end_of_week = date.fromisocalendar(year, week_num, 7)    # 周日

    # 获取这一周内的所有交易日
    schedule = calendar.schedule(start_date=start_of_week, end_date=end_of_week)
    trading_days = schedule.index.date.tolist()

    if not trading_days:
        return False

    last_trading_day_of_week = trading_days[-1]

    return input_date == last_trading_day_of_week


def is_monthly_last_trading_date(input_date):
    if isinstance(input_date, str):
        input_date = pd.to_datetime(input_date).date()
    elif isinstance(input_date, pd.Timestamp):
        input_date = input_date.date()

    # 找到 input_date 所在月份的第一天和最后一天
    first_day_of_month = input_date.replace(day=1)
    next_month = first_day_of_month + timedelta(days=32)
    last_day_of_month = next_month.replace(day=1) - timedelta(days=1)

    # 获取该月的所有交易日
    schedule = calendar.schedule(start_date=first_day_of_month, end_date=last_day_of_month)
    trading_days = schedule.index.date.tolist()

    if not trading_days:
        return False

    last_trading_day_of_month = trading_days[-1]

    return input_date == last_trading_day_of_month


In [19]:
input_date = datetime.today().date() + timedelta(days=2) 
print(input_date)


print(is_monthly_last_trading_date(input_date))
print(is_weekly_last_trading_date(input_date))

2025-05-30
True
True


In [28]:
import yfinance as yf

proxies = {'http': 'http://127.0.0.1:10086', 'https': 'http://127.0.0.1:10086'}

ticker_data = yf.download('QQQ', start='2025-01-01', end='2025-05-29',
				interval='1d', proxy=proxies, auto_adjust=True)
ticker_data

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,QQQ,QQQ,QQQ,QQQ,QQQ
Date,,,,,
2025-01-02,509.470245,515.870704,504.956956,513.534162,36389800
2025-01-03,517.807800,518.876214,511.766822,512.585547,29059500
2025-01-06,523.758911,527.133883,521.252700,523.239727,36109700
2025-01-07,514.412842,525.206742,512.515708,524.807374,36690300
2025-01-08,514.502747,516.150253,509.809733,514.313027,30777800
...,...,...,...,...,...
2025-05-21,513.039978,523.000000,511.239990,516.270020,71292800
2025-05-22,514.000000,517.710022,512.299988,513.659973,55607200


In [26]:
ticker_data = yf.download('SPY', start='2025-05-01', end='2025-05-29',
				interval='1d', proxy=proxies, auto_adjust=True)
ticker_data

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2025-05-01,558.469971,564.070007,557.859985,560.369995,63186100
2025-05-02,566.760010,568.380005,562.380005,564.729980,60717300
2025-05-05,563.510010,566.650024,561.700012,562.570007,38659200
2025-05-06,558.799988,563.349976,556.960022,557.929993,48264700
2025-05-07,561.150024,563.820007,556.039978,560.150024,55588000
2025-05-08,565.059998,570.309998,561.700012,565.239990,65130800
2025-05-09,564.340027,567.500000,562.760010,566.479980,37603400
2025-05-12,582.989990,583.000000,577.039978,581.469971,78993600


In [25]:
ticker_data = yf.download('TSLA', start='2025-05-01', end='2025-05-29',
				interval='1d', proxy=proxies, auto_adjust=False)
ticker_data

[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,TSLA,TSLA,TSLA,TSLA,TSLA,TSLA
Date,,,,,,
2025-05-01,280.519989,280.519989,290.869995,279.809998,280.010010,99659000
2025-05-02,287.209991,287.209991,294.779999,279.809998,284.899994,114454700
2025-05-05,280.260010,280.260010,284.850006,274.399994,284.570007,94618900
2025-05-06,275.350006,275.350006,277.730011,271.350006,273.109985,76715800
2025-05-07,276.220001,276.220001,277.920013,271.000000,276.880005,71592600
2025-05-08,284.820007,284.820007,289.799988,279.410004,279.630005,97539400
2025-05-09,298.260010,298.260010,307.040009,290.000000,290.209991,132387800
2025-05-12,318.380005,318.380005,322.209991,311.500000,321.989990,112826700


In [ ]:
import pandas as pd

In [48]:
current_date = pd.Timestamp.today() - timedelta(hours=14) 
print(current_date)

delta_days = 1 if current_date.hour <= 20 else 0     # UTC时间20点对应NY时间16点
print(delta_days)

previous_business_day = current_date - pd.offsets.BusinessDay(n=delta_days)
previous_business_day = previous_business_day.strftime('%Y-%m-%d')
print(previous_business_day)

2025-05-28 21:03:16.906474
0
2025-05-28


In [57]:
API_KEY = 'BSLRAR2IZ893LHH2'
ticker_id = "TSLA"
interval = "5min"  # 1min, 5min, 15min, 30min, 60min

In [ ]:
"""
This API returns raw (as-traded) daily time series 
- (date, daily open, daily high, daily low, daily close, daily volume) of the global equity specified, 
- covering 20+ years of historical data. 
- The OHLCV data is sometimes called "candles" in finance literature. 
- If you are also interested in split/dividend-adjusted data, please use the Daily Adjusted API, 
- which covers adjusted close values and historical split and dividend events.
"""



https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=TSLA&apikey=BSLRAR2IZ893LHH2&datatype=csv
https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol=TSLA&apikey=BSLRAR2IZ893LHH2&datatype=csv


https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol=TSLA&apikey=BSLRAR2IZ893LHH2&datatype=csv
https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY_ADJUSTED&symbol={ticker_id}&apikey={API_KEY}&datatype=csv


In [65]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&apikey=demo'
r = requests.get(url)
data = r.json()
print(data)

{'Meta Data': {'1. Information': 'Intraday (5min) open, high, low, close prices and volume', '2. Symbol': 'IBM', '3. Last Refreshed': '2025-05-29 19:55:00', '4. Interval': '5min', '5. Output Size': 'Compact', '6. Time Zone': 'US/Eastern'}, 'Time Series (5min)': {'2025-05-29 19:55:00': {'1. open': '258.6800', '2. high': '258.6800', '3. low': '258.6800', '4. close': '258.6800', '5. volume': '15'}, '2025-05-29 19:45:00': {'1. open': '258.3100', '2. high': '258.3100', '3. low': '258.3100', '4. close': '258.3100', '5. volume': '10'}, '2025-05-29 19:40:00': {'1. open': '258.6900', '2. high': '258.6900', '3. low': '258.6900', '4. close': '258.6900', '5. volume': '10'}, '2025-05-29 19:20:00': {'1. open': '258.8699', '2. high': '258.9900', '3. low': '256.8600', '4. close': '258.9900', '5. volume': '5'}, '2025-05-29 19:05:00': {'1. open': '258.8700', '2. high': '258.8700', '3. low': '258.8700', '4. close': '258.8700', '5. volume': '13'}, '2025-05-29 19:00:00': {'1. open': '258.6900', '2. high': 

In [ ]:
import pandas as pd

df = pd.read_csv("https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=TSLA&apikey=BSLRAR2IZ893LHH2&datatype=csv")


In [67]:
print(df.head())

    timestamp    open    high     low   close     volume
0  2025-05-29  365.29  367.71  356.00  358.43   87372938
1  2025-05-28  364.84  365.00  355.91  356.90   91404309
2  2025-05-27  347.35  363.79  347.32  362.89  120146414
3  2025-05-23  337.92  343.18  333.21  339.34   84654818
4  2025-05-22  331.90  347.27  331.39  341.04   97113416


In [70]:
ticker_id = 'TSLA'
API_KEY = "IA5j5a8TjMEEQOV9RHe8s7oaKVboVeCp"
print(ticker_id)
url = f"https://financialmodelingprep.com/stable/historical-chart/5min?symbol={ticker_id}&apikey={API_KEY}"
print(url)
from urllib.request import urlopen
import certifi
import json


response = urlopen(url, cafile=certifi.where())
data = response.read().decode("utf-8")
data = json.loads(data)
print(data)


TSLA
https://financialmodelingprep.com/stable/historical-chart/5min?symbol=TSLA&apikey=IA5j5a8TjMEEQOV9RHe8s7oaKVboVeCp


/var/folders/td/16v_twyd2fq9rhdl5vysffbw0000gn/T/ipykernel_84607/1901761326.py:11: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url, cafile=certifi.where())


[{'date': '2025-05-29 15:55:00', 'open': 358.455, 'low': 358.26, 'high': 358.89, 'close': 358.39, 'volume': 1692076}, {'date': '2025-05-29 15:50:00', 'open': 358.8, 'low': 357.67, 'high': 359.5, 'close': 358.3955, 'volume': 1189736}, {'date': '2025-05-29 15:45:00', 'open': 357.535, 'low': 357.535, 'high': 358.8, 'close': 358.62, 'volume': 902064}, {'date': '2025-05-29 15:40:00', 'open': 357.59, 'low': 357.25, 'high': 358.09, 'close': 357.57, 'volume': 1057456}, {'date': '2025-05-29 15:35:00', 'open': 357.94, 'low': 357.4922, 'high': 358.0393, 'close': 357.6, 'volume': 580368}, {'date': '2025-05-29 15:30:00', 'open': 358.12, 'low': 357.785, 'high': 358.71, 'close': 357.96, 'volume': 650946}, {'date': '2025-05-29 15:25:00', 'open': 359.11, 'low': 358.035, 'high': 359.3491, 'close': 358.11, 'volume': 676575}, {'date': '2025-05-29 15:20:00', 'open': 358.6, 'low': 358.6, 'high': 359.26, 'close': 359.12, 'volume': 689418}, {'date': '2025-05-29 15:15:00', 'open': 357.6721, 'low': 357.6721, 'h

In [69]:
ticker_id = 'TSLA'
API_KEY = "IA5j5a8TjMEEQOV9RHe8s7oaKVboVeCp"

url = f"https://financialmodelingprep.com/stable/historical-price-eod/non-split-adjusted?symbol={ticker_id}&apikey={API_KEY}"
response = urlopen(url, cafile=certifi.where())
data = response.read().decode("utf-8")
data = json.loads(data)
df = pd.DataFrame(data)
print(df)

/var/folders/td/16v_twyd2fq9rhdl5vysffbw0000gn/T/ipykernel_84607/1639344083.py:5: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url, cafile=certifi.where())


     symbol        date  adjOpen  adjHigh  adjLow  adjClose     volume
0      TSLA  2025-05-29   365.29   367.71  356.00    358.43   88545666
1      TSLA  2025-05-28   364.84   365.00  355.91    356.90   91404309
2      TSLA  2025-05-27   347.35   363.79  347.32    362.89  120146414
3      TSLA  2025-05-23   337.92   343.18  333.21    339.34   84654818
4      TSLA  2025-05-22   331.90   347.27  331.39    341.04   97113416
...     ...         ...      ...      ...     ...       ...        ...
1251   TSLA  2020-06-05   877.80   886.50  866.25    885.60    7811900
1252   TSLA  2020-06-04   889.95   895.80  858.45    864.45    8887700
1253   TSLA  2020-06-03   888.15   897.90  880.05    882.90    7949500
1254   TSLA  2020-06-02   894.75   908.70  871.05    881.55   13565600
1255   TSLA  2020-06-01   858.00   898.95  854.10    898.05   14939500

[1256 rows x 7 columns]


In [ ]:
df = df.rename(columns={
"date": "Datetime",
"open": "Open",
"low": "Low",
"high": "High",
"close": "Close",
"volume": "Volume",
})

df['ticker_id'] = ticker_id
df['Adj High'] = df['High']
df['Adj Close'] = df['Close']
df['Adj Low'] = df['Low']
df['Adj Open'] = df['Open']


In [2]:
import requests
import json
import os # 用于处理文件路径

# --- 配置参数 ---
BASE_URL = "https://engine.invesdo.co:80/api/web/market/overview"
PAYLOAD = {
    "user_id": 123,
    "user_name": "test_user",
    "user_language": "en"
}
HEADERS = {
    "Content-Type": "application/json"
}

CA_CERT_PATH = "/Users/liuda/Library/CloudStorage/Dropbox/shareit/code/trading/Agent-engines-python/config/pem_v2/engine.invesdo.co.cer" # <--- 修改这里!

response = requests.post(
	BASE_URL,
	headers=HEADERS,
	data=json.dumps(PAYLOAD), # 将 Python 字典转换为 JSON 字符串
	verify=CA_CERT_PATH,      # 启用 SSL 证书验证，并指定 CA 证书路径
	timeout=600
)

print(response)
print(response.text)

SSLError: HTTPSConnectionPool(host='engine.invesdo.co', port=80): Max retries exceeded with url: /api/web/market/overview (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1017)')))

In [17]:
import pandas as pd

ticker_data = pd.read_csv("/Users/liuda/Desktop/MSTR.csv")
ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
ticker_data.set_index('Date', inplace=True)

print(ticker_data.shape)
import talib
talib.EMA(ticker_data['Adj Close'], timeperiod=144)

(364, 236)


Date
2024-06-05           NaN
2024-06-06           NaN
2024-06-07           NaN
2024-06-08           NaN
2024-06-09           NaN
                 ...    
2025-05-30    336.736891
2025-05-31    337.182727
2025-06-01    337.622414
2025-06-02    338.100312
2025-06-03    338.780721
Length: 364, dtype: float64

In [16]:
import pandas as pd

ticker_data = pd.read_csv("/Users/liuda/Desktop/MSTR.csv")
ticker_data = ticker_data[ticker_data['Date'] >= '2024-06-10']

ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
ticker_data.set_index('Date', inplace=True)
print(ticker_data.shape)

import talib
talib.EMA(ticker_data['Adj Close'], timeperiod=144)

(359, 236)


Date
2024-06-10           NaN
2024-06-11           NaN
2024-06-12           NaN
2024-06-13           NaN
2024-06-14           NaN
                 ...    
2025-05-30    336.561724
2025-05-31    337.009976
2025-06-01    337.452045
2025-06-02    337.932293
2025-06-03    338.615020
Length: 359, dtype: float64